# Step 1: Master Relational Joins, NLTK VADER NLP & Spatial Coordinates
**MSc Data Science Thesis — University of Wolverhampton**

### Objective & Methodological Alignment
Consolidate 7 raw Olist e-commerce relational tables (`orders`, `order_items`, `customers`, `products`, `reviews`, `geolocation`, `translation`).

### Features Implemented:
1. **NLTK VADER NLP Sentiment Analysis**: Computes continuous compound polarity scores (`sentiment_score`) on Portuguese/English review messages.
2. **Spatial Coordinate Collapse**: Mean-aggregates latitude/longitude coordinates by `geolocation_zip_code_prefix` to attach `customer_lat` and `customer_lon` without row expansion.
3. **Sentiment-Validated Target Proxy Definition**:
   $$\text{is\_returned}_i = \begin{cases} 1 & \text{if } \text{order\_status}_i = \text{'canceled'} \lor \text{review\_score}_i \le 2.0 \lor \text{sentiment\_score}_i \le -0.5 \\ 0 & \text{otherwise} \end{cases}$$
4. **Output Checkpoint**: `consolidated_return_data.csv`.

In [3]:
import pandas as pd
import numpy as np
import nltk
from nltk.sentiment.vader import SentimentIntensityAnalyzer

# Initialize NLTK VADER Sentiment Analyzer for thesis text evaluation layer
nltk.download('vader_lexicon', quiet=True)
sia = SentimentIntensityAnalyzer()

print("Step 1: Loading raw datasets from local workspace directory...")
# Load datasets using verified character decoders
customers = pd.read_csv('olist_customers_dataset.csv', encoding='latin-1')
orders = pd.read_csv('olist_orders_dataset.csv', encoding='latin-1')
items = pd.read_csv('olist_order_items_dataset.csv', encoding='latin-1')
products = pd.read_csv('olist_products_dataset.csv', encoding='latin-1')
reviews = pd.read_csv('olist_order_reviews_dataset.csv', encoding='latin-1')
geolocation = pd.read_csv('olist_geolocation_dataset.csv', encoding='latin-1')
# BOM FORMAT FIX: Read the translation file with utf-8-sig to clear 'ï»¿' symbols
translation = pd.read_csv('product_category_name_translation.csv', encoding='utf-8-sig')

# Clean hidden whitespaces from columns across all data frames
for df_obj in [customers, orders, items, products, reviews, translation, geolocation]:
    df_obj.columns = df_obj.columns.str.strip()

print("Executing Master Relational Joins...")
df = pd.merge(orders, items, on='order_id', how='left')
df = pd.merge(df, customers, on='customer_id', how='left')
df = pd.merge(df, products, on='product_id', how='left')
df = pd.merge(df, translation, on='product_category_name', how='left')

print("Collapsing duplicate geolocation data coordinates to prevent row expansion...")
# Deduplicate spatial table parameters via mean aggregation
geo_collapsed = geolocation.groupby('geolocation_zip_code_prefix').agg({
    'geolocation_lat': 'mean',
    'geolocation_lng': 'mean'
}).reset_index()

# Map customer geographical coordinates safely
df = pd.merge(df, geo_collapsed, left_on='customer_zip_code_prefix', right_on='geolocation_zip_code_prefix', how='left')
df.rename(columns={'geolocation_lat': 'customer_lat', 'geolocation_lng': 'customer_lon'}, inplace=True)
df.drop(columns=['geolocation_zip_code_prefix'], errors='ignore', inplace=True)

print("Running text sentiment profiling utilizing NLTK VADER NLP...")
reviews['review_comment_message_english'] = reviews['review_comment_message_english'].replace('#VALUE!', 'No comment').fillna('No comment')

def compute_vader_compound(text):
    if text == 'No comment': return 0.0
    return sia.polarity_scores(str(text))['compound']

reviews['sentiment_score'] = reviews['review_comment_message_english'].apply(compute_vader_compound)

# Aggregate multiple customer review metrics per unique order ID
reviews_clean = reviews.groupby('order_id').agg({
    'review_score': 'mean',
    'sentiment_score': 'mean',
    'review_comment_message_english': 'first'
}).reset_index()

df = pd.merge(df, reviews_clean, on='order_id', how='left')

print("Constructing sentiment-validated proxy label for machine learning target...")
# Flagged as 1 if order canceled OR VADER compound text score <= -0.5
df['is_returned'] = ((df['order_status'] == 'canceled') | (df['review_score'] <= 2) | (df['sentiment_score'] <= -0.5)).astype(int)

# Export structural intermediate tracking file
df.to_csv('consolidated_return_data.csv', index=False)
print(f"Success! Consolidated dataset generated with shape: {df.shape}\n")

Step 1: Loading raw datasets from local workspace directory...
Executing Master Relational Joins...
Collapsing duplicate geolocation data coordinates to prevent row expansion...
Running text sentiment profiling utilizing NLTK VADER NLP...
Constructing sentiment-validated proxy label for machine learning target...
Success! Consolidated dataset generated with shape: (113425, 33)

